# Payments & Product Performance Analysis

This notebook explores how customers pay, which products generate the most buyer activity, and how performance differs by education format.

### Business questions
- Which payment methods are most common among confirmed buyers?
- How do recorded payments differ by payment type?
- Which products attract the most buyers and recorded revenue?
- How does performance differ between Morning and Evening formats?
- Does payment preference vary by product?

> **Interpretation note:** payment and education fields are most complete at later funnel stages, so the analysis is primarily descriptive rather than causal.


In [ ]:
from pathlib import Path
import sys

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "helpers.py").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from helpers import colors

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

pd.options.display.max_columns = None


## Load Processed Data

In [ ]:
deals = pd.read_pickle(PROCESSED_DIR / 'deals_clean.pkl')
buyers = pd.read_pickle(PROCESSED_DIR / 'buyers.pkl')

## 1. Payment Types Among Confirmed Buyers

In [ ]:
# Analyze deals with confirmed Payment Done status
stage_done = deals[deals['stage'] == 'Payment Done']
payment_counts = stage_done['payment_type'].value_counts()

plt.figure(figsize=(5, 5))

plt.pie(payment_counts, labels=payment_counts.index, autopct='%1.1f%%',startangle=90, colors=[colors['accent'], '#10B981', '#F59E0B', '#94A3B8'], wedgeprops={'width': 0.5, 'edgecolor': 'white'})

plt.title('Payment Type Distribution')
plt.tight_layout()
plt.show()

In [ ]:
# Compare recorded initial payments and potential contract value
payment_stats = stage_done.groupby('payment_type').agg(revenue_actual=('initial_amount_paid', 'sum'), potential_revenue=('offer_total_amount', 'sum'))
payment_stats

### Payment Insights

- **Recurring Payments** are the dominant payment option among confirmed buyers.
- **One Payment** remains an important alternative and contributes meaningful recorded revenue.
- **Reservation** is rarely used in the analyzed data.

Because `payment_type` is a manually maintained CRM field, results should be interpreted together with recorded payment amounts rather than as a perfect transaction ledger.

## 2. Product and Education-Format Performance

### 2.1 Product Mix

In [ ]:
# Product distribution across leads/deals

In [ ]:
deals['product'].value_counts()

Two very small product categories (`Find yourself in IT` and `Data Analytics`) are excluded from the main product comparison to avoid unstable rankings based on very small samples.

In [ ]:
# Exclude very small product categories from the main comparison
products = deals[~deals['product'].isin(['Find yourself in IT', 'Data Analytics'])]

# Count deals by product
product_counts = products['product'].value_counts()

plt.figure(figsize=(5, 5))

plt.pie(product_counts, labels=product_counts.index, autopct='%1.1f%%', startangle=90, colors=[colors['accent'], '#10B981', '#F59E0B'], wedgeprops={'width': 0.5, 'edgecolor': 'white'})

plt.title('Share of Deals by Product')
plt.tight_layout()
plt.show()

In [ ]:
# Product performance among confirmed buyers

In [ ]:
# Recorded revenue proxy = initial_amount_paid
product_stats = (products[products['is_buyer']].groupby('product').agg(buyers=('is_buyer', 'sum'), revenue=('initial_amount_paid', 'sum')))
product_stats['revenue_per_buyer'] = (product_stats['revenue'] / product_stats['buyers']).round(2)
product_stats

In [ ]:
plot_product = (product_stats.sort_values('revenue', ascending=False).reset_index())

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 1. Revenue
sns.barplot(data=plot_product, y='revenue', x='product', color=colors['accent'], ax=axes[0])

axes[0].bar_label(axes[0].containers[0], labels=[f'€{x:,.0f}' for x in plot_product['revenue']], padding=3)

axes[0].set_title('Revenue')
axes[0].set_xlabel('')
axes[0].set_ylabel('')


# 2. Revenue per buyer
sns.barplot(data=plot_product, y='revenue_per_buyer', x='product', color=colors['accent'], ax=axes[1])

axes[1].bar_label(axes[1].containers[0], labels=[f'€{x:,.0f}' for x in plot_product['revenue_per_buyer']], padding=3)

axes[1].set_title('Revenue per buyer')
axes[1].set_xlabel('')
axes[1].set_ylabel('')


# 3. Buyers
sns.barplot(data=plot_product, y='buyers', x='product', color=colors['accent'], ax=axes[2])

axes[2].bar_label(axes[2].containers[0], fmt='%.0f', padding=3)

axes[2].set_title('Buyers')
axes[2].set_xlabel('')
axes[2].set_ylabel('')

plt.suptitle('Product Type Performance')
plt.tight_layout()
plt.show()

In [ ]:
# Product performance split by education format

In [ ]:
buyers['education_type'].value_counts(dropna=False)

Seven buyers have no recorded `education_type`, so product-by-format comparisons do not include these records.

In [ ]:
product_stats = (products[products['is_buyer']].groupby(['product', 'education_type']).agg(buyers=('is_buyer', 'sum'),revenue=('initial_amount_paid', 'sum')))

product_stats['revenue_per_buyer'] = (product_stats['revenue'] / product_stats['buyers'])

product_stats

In [ ]:
plot_product = (product_stats.reset_index().sort_values('revenue', ascending=False))

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

palette = {'Morning': colors['accent'],'Evening': '#64748B'}

# 1. Buyers
sns.barplot(data=plot_product, x='product', y='buyers', hue='education_type', palette=palette, ax=axes[0])

axes[0].set_title('Buyers by Product and Education Type')
axes[0].set_xlabel('')
axes[0].set_ylabel('Buyers')
axes[0].tick_params(axis='x', rotation=20)

# 2. Revenue
sns.barplot(data=plot_product, x='product', y='revenue', hue='education_type', palette=palette, ax=axes[1])

axes[1].set_title('Revenue by Product and Education Type')
axes[1].set_xlabel('')
axes[1].set_ylabel('Revenue (€)')
axes[1].tick_params(axis='x', rotation=20)

# 3. Revenue per Buyer
sns.barplot(data=plot_product, x='product', y='revenue_per_buyer', hue='education_type', palette=palette, ax=axes[2])

axes[2].set_title('Revenue per Buyer by Product and Education Type')
axes[2].set_xlabel('')
axes[2].set_ylabel('Revenue per Buyer (€)')
axes[2].tick_params(axis='x', rotation=20)

# Value labels
for container in axes[0].containers:
    axes[0].bar_label(container, fmt='%.0f', padding=3)

for container in axes[1].containers:
    axes[1].bar_label(
        container, labels=[f'€{v:,.0f}' if v > 0 else '' for v in container.datavalues], padding=3)

for container in axes[2].containers:
    axes[2].bar_label(
        container,
        labels=[f'€{v:,.0f}' if v > 0 else '' for v in container.datavalues],
        padding=3)

# Keep the legend only on the first chart
axes[1].legend_.remove()
axes[2].legend_.remove()

plt.suptitle(
    'Product Performance by Education Type',
    fontsize=16,
    fontweight='bold',
    y=1.03
)

plt.tight_layout()
plt.show()

### Product Economics Insights

- **Digital Marketing** is the largest product by buyer volume and recorded initial payments.
- Morning groups outperform Evening groups in buyer volume and total recorded initial payments across products where both formats are present.
- Revenue per buyer differs by product and format, reflecting differences in course pricing and payment structure.
- **Web Developer** is represented only in the Morning format in the analyzed buyer data.

## 3. Product × Payment Method

In [ ]:
# Product × Payment Type cross-tabulation
cross_pr_pt = pd.crosstab(buyers['product'],buyers['payment_type'], margins=True, margins_name='Total')
cross_pr_pt

In [ ]:
# Row percentages: payment preference within each product
cross_pr_pt_pct = pd.crosstab(buyers['product'], buyers['payment_type'], normalize='index').round(3) * 100

cross_pr_pt_pct

In [ ]:
plt.figure(figsize=(8, 4))

sns.heatmap(cross_pr_pt_pct, annot=True, fmt='.0f', cmap='Blues', linewidths=0.5)

plt.title('Preferred Payment Type by Product (%)')
plt.xlabel('Payment Type')
plt.ylabel('Product')

plt.tight_layout()
plt.show()

In [ ]:
# Compare median offer value by product and education format
buyers.groupby(by=['product', 'education_type'])['offer_total_amount'].median()

### Product × Payment Insights

- **Digital Marketing** and **UX/UI Design** are predominantly paid through recurring payments.
- **Web Developer** has a much more balanced split between one-time and recurring payments.
- The pattern is consistent with differences in course price: more expensive programs are more frequently associated with installment-style payments.

**Sales implication:** payment options can be tailored by product rather than applying the same default recommendation to every course.

### Education-Format Insights

- The **Morning** format represents the majority of confirmed buyers and generates most of the observed revenue.
- Evening programs are generally lower priced, which contributes to lower revenue per buyer.
- `education_type` is mainly populated at final deal stages, so a true Morning-vs-Evening conversion comparison cannot be calculated reliably from this field alone. Earlier-stage preference data would be required.